In [ ]:
import os
import matplotlib.pyplot as plt

# Define the paths to the image directories
# Paths need to be modified locally after dataset is downloaded from Kaggle.

train_dr_path = 'Diagnosis of Diabetic Retinopathy/train/DR'
train_no_dr_path = 'Diagnosis of Diabetic Retinopathy/train/No_DR'

test_dr_path = 'Diagnosis of Diabetic Retinopathy/test/DR'
test_no_dr_path = 'Diagnosis of Diabetic Retinopathy/test/No_DR'

valid_dr_path = 'Diagnosis of Diabetic Retinopathy/valid/DR'
valid_no_dr_path = 'Diagnosis of Diabetic Retinopathy/valid/No_DR'

# Function to count the number of files in a directory
def count_files(directory):
    return len([filename for filename in os.listdir(directory) if os.path.isfile(os.path.join(directory, filename))])

# Count the number of images in each category
train_dr_count = count_files(train_dr_path)
train_no_dr_count = count_files(train_no_dr_path)

test_dr_count = count_files(test_dr_path)
test_no_dr_count = count_files(test_no_dr_path)

valid_dr_count = count_files(valid_dr_path)
valid_no_dr_count = count_files(valid_no_dr_path)

# Plotting the bar chart
categories = ['Train DR', 'Train No DR', 'Test DR', 'Test No DR', 'Valid DR', 'Valid No DR']
counts = [train_dr_count, train_no_dr_count, test_dr_count, test_no_dr_count, valid_dr_count, valid_no_dr_count]

plt.bar(categories, counts, color=['blue', 'green', 'red', 'purple', 'orange', 'brown'])
plt.xlabel('Categories')
plt.ylabel('Number of Images')
plt.title('Distribution of Images in Diabetic Retinopathy Dataset')
plt.show()

We now try to see a few example images

In [ ]:
from PIL import Image
import random

# Function to get the size of an image
def get_image_size(image_path):
    with Image.open(image_path) as img:
        return img.size

# Function to plot random images from both categories in a subplot
def plot_random_images(category1_path, category1_name, category2_path, category2_name, num_images=4):
    fig, axes = plt.subplots(2, num_images // 2, figsize=(15, 10))
    axes = axes.flatten()

    for i in range(num_images):
        if i < num_images // 2:
            category_path = category1_path
            category_name = category1_name
        else:
            category_path = category2_path
            category_name = category2_name

        file_list = [f for f in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, f))]
        random_file = random.choice(file_list)
        image_path = os.path.join(category_path, random_file)

        image_size = get_image_size(image_path)
        axes[i].imshow(Image.open(image_path))
        axes[i].set_title(f"{category_name} Image\nSize: {image_size}")
        axes[i].axis('off')

    plt.show()

# Plot random images in a subplot
plot_random_images(train_dr_path, "DR", train_no_dr_path, "No DR", num_images=4)

Lets build a Dataloader for we would be using for training the next few models

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define data transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])



# Create datasets
train_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/train', transform=transform)
test_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/test', transform=transform)
valid_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/valid', transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)


In [ ]:
!pip install seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision import models
from sklearn.metrics import roc_curve, auc, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

We first train the model with ResNet50 and Plot the ROC and Confusion Matrix

In [ ]:
# Load the pre-trained ResNet-50 model
resnet_model = models.resnet50(pretrained=True)

# Specify the device, e.g., 'cuda' for GPU or 'cpu' for CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Freeze all the layers in the pre-trained model
for param in resnet_model.parameters():
    param.requires_grad = False

# Modify the fully connected layer to match the number of classes in your dataset
num_classes = len(train_dataset.classes)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# Move the model to the GPU if available
resnet_model = resnet_model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.fc.parameters(), lr=0.001)

# Variables for storing best accuracy and the corresponding model
best_accuracy_resnet = 0.0
best_model_resnet = None

# Lists for storing training and validation accuracies for plotting
train_accuracies = []
valid_accuracies = []

# Early stopping parameters
early_stopping_patience = 10
early_stopping_counter = 0
best_valid_loss = float('inf')

# Training loop for 100 epochs (with early stopping)
num_epochs = 100

for epoch in range(num_epochs):
    resnet_model.train()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Evaluate the model on the training set
    resnet_model.eval()
    train_correct = 0
    train_total = 0

    with torch.no_grad():
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = resnet_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

    train_accuracy = train_correct / train_total

    # Evaluate the model on the validation set
    resnet_model.eval()
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = resnet_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = valid_correct / valid_total

    # Save the model with the best accuracy on the validation set
    if valid_accuracy > best_accuracy_resnet:
        best_accuracy_resnet = valid_accuracy
        best_model_resnet = resnet_model.state_dict()
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    # Store accuracies for plotting
    train_accuracies.append(train_accuracy)
    valid_accuracies.append(valid_accuracy)

    # Print and save model checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Accuracy: {train_accuracy:.4f}, Valid Accuracy: {valid_accuracy:.4f}')

    # Early stopping
    if early_stopping_counter >= early_stopping_patience:
        print("Early stopping! No improvement for {} epochs.".format(early_stopping_patience))
        break

# Plot the accuracy change with epochs
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, len(valid_accuracies) + 1), valid_accuracies, label='Valid Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.legend()
plt.show()

# Load the best model and evaluate it on the test set
resnet_model.load_state_dict(best_model_resnet)
resnet_model.eval()
test_correct = 0
test_total = 0
all_labels = []
all_predictions = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = resnet_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(torch.nn.functional.softmax(outputs, dim=1)[:, 1].cpu().numpy())

test_accuracy_resnet = test_correct / test_total
print(f'Best Accuracy on Test Set (ResNet): {100 * test_accuracy_resnet:.2f}%')

# Convert the lists to numpy arrays
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# Plot ROC Curve for ResNet
fpr_resnet, tpr_resnet, thresholds_resnet = roc_curve(all_labels, all_predictions)
roc_auc_resnet = auc(fpr_resnet, tpr_resnet)

plt.figure()
plt.plot(fpr_resnet, tpr_resnet, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_resnet:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve (ResNet)')
plt.legend(loc="lower right")
plt.show()

# Plot Confusion Matrix for ResNet
cm_resnet = confusion_matrix(all_labels, (all_predictions > 0.5).astype(int))
plt.figure(figsize=(5, 4))
sns.heatmap(cm_resnet, annot=True, fmt="d", cmap="Blues", xticklabels=["No DR", "DR"], yticklabels=["No DR", "DR"])
plt.title("Confusion Matrix (ResNet)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
resnet_sensitivity = 106/109
resnet_specificity = 115/122

Next, we will use effiecientnet

In [ ]:
!pip install efficientnet_pytorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from efficientnet_pytorch import EfficientNet
from sklearn.metrics import roc_curve, auc, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

The same dataloader has been used, but with the efficientnet pretrained model now

In [ ]:
# Load the pre-trained EfficientNet model
effnet_model = EfficientNet.from_pretrained('efficientnet-b2', num_classes=len(train_dataset.classes))

# Move the model to the GPU if available
effnet_model = effnet_model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(effnet_model.parameters(), lr=0.001)

# Variables for storing best accuracy and the corresponding model
best_accuracy_effnet = 0.0
best_model_effnet = None

# Lists for storing training and validation accuracies for plotting
train_accuracies_effnet = []
valid_accuracies_effnet = []

# Early stopping parameters
early_stopping_patience = 10
early_stopping_counter = 0
best_valid_loss_effnet = float('inf')

# Training loop for 100 epochs (with early stopping)
num_epochs = 100

for epoch in range(num_epochs):
    effnet_model.train()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = effnet_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Evaluate the model on the training set
    effnet_model.eval()
    train_correct = 0
    train_total = 0

    with torch.no_grad():
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = effnet_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

    train_accuracy_effnet = train_correct / train_total

    # Evaluate the model on the validation set
    effnet_model.eval()
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = effnet_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy_effnet = valid_correct / valid_total

    # Save the model with the best accuracy on the validation set
    if valid_accuracy_effnet > best_accuracy_effnet:
        best_accuracy_effnet = valid_accuracy_effnet
        best_model_effnet = effnet_model.state_dict()
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    # Store accuracies for plotting
    train_accuracies_effnet.append(train_accuracy_effnet)
    valid_accuracies_effnet.append(valid_accuracy_effnet)

    # Print and save model checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Accuracy: {train_accuracy_effnet:.4f}, Valid Accuracy: {valid_accuracy_effnet:.4f}')

    # Early stopping
    if early_stopping_counter >= early_stopping_patience:
        print("Early stopping! No improvement for {} epochs.".format(early_stopping_patience))
        break

# Plot the accuracy change with epochs for EfficientNet
plt.plot(range(1, len(train_accuracies_effnet) + 1), train_accuracies_effnet, label='Train Accuracy (EfficientNet)')
plt.plot(range(1, len(valid_accuracies_effnet) + 1), valid_accuracies_effnet, label='Valid Accuracy (EfficientNet)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Over Epochs (EfficientNet)')
plt.legend()
plt.show()

# Load the best model and evaluate it on the test set for EfficientNet
effnet_model.load_state_dict(best_model_effnet)
effnet_model.eval()
test_correct_effnet = 0
test_total_effnet = 0
all_labels_effnet = []
all_predictions_effnet = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = effnet_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total_effnet += labels.size(0)
        test_correct_effnet += (predicted == labels).sum().item()

        all_labels_effnet.extend(labels.cpu().numpy())
        all_predictions_effnet.extend(torch.nn.functional.softmax(outputs, dim=1)[:, 1].cpu().numpy())

test_accuracy_effnet = test_correct_effnet / test_total_effnet
print(f'Best Accuracy on Test Set (EfficientNet): {100 * test_accuracy_effnet:.2f}%')

# Convert the lists to numpy arrays for EfficientNet
all_labels_effnet = np.array(all_labels_effnet)
all_predictions_effnet = np.array(all_predictions_effnet)

# Plot ROC Curve for EfficientNet
fpr_effnet, tpr_effnet, thresholds_effnet = roc_curve(all_labels_effnet, all_predictions_effnet)
roc_auc_effnet = auc(fpr_effnet, tpr_effnet)

plt.figure()
plt.plot(fpr_effnet, tpr_effnet, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_effnet:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve (EfficientNet)')
plt.legend(loc="lower right")
plt.show()

# Plot Confusion Matrix for EfficientNet
cm_effnet = confusion_matrix(all_labels_effnet, (all_predictions_effnet > 0.5).astype(int))
plt.figure(figsize=(5, 4))
sns.heatmap(cm_effnet, annot=True, fmt="d", cmap="Blues", xticklabels=["No DR", "DR"], yticklabels=["No DR", "DR"])
plt.title("Confusion Matrix (EfficientNet)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
effnet_sensitivity = 110/112
effnet_specificity = 116/119

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import mobilenet_v3_small
from sklearn.metrics import roc_curve, auc, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Our Hypothesis however was that, vision transformers will have significantly better performance comparing to cnn based models, so for this we would use pretrained ViT model

In [ ]:
pip install pytorch_pretrained_vit

In [ ]:
from pytorch_pretrained_vit import ViT
vit_model = ViT('B_16_imagenet1k', pretrained=True)

the input image size of this model is different, thus we would need to change the dataloader

In [ ]:
print(vit_model.image_size)

In [ ]:
# Define data transformations
transform = transforms.Compose([
    transforms.Resize(vit_model.image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Create datasets
train_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/train', transform=transform)
test_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/test', transform=transform)
valid_dataset = ImageFolder(root='Diagnosis of Diabetic Retinopathy/valid', transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)




Loading the model to device and setting parameters

In [ ]:
# Specify the device, e.g., 'cuda' for GPU or 'cpu' for CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Freeze all the layers in the pre-trained model
for param in vit_model.parameters():
    param.requires_grad = False

# Modify the fully connected layer to match the number of classes in your dataset
num_classes = len(train_dataset.classes)
vit_model.fc = nn.Linear(vit_model.fc.in_features, num_classes)

# Move the model to the GPU if available
vit_model = vit_model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vit_model.fc.parameters(), lr=0.001)

# Variables for storing best accuracy and the corresponding model
best_accuracy_vit = 0.0
best_model_vit = None


Setting early stoppage with a possible 100 epochs of training, just like the previous models

In [ ]:
# Lists for storing training and validation accuracies for plotting
train_accuracies = []
valid_accuracies = []

# Early stopping parameters
early_stopping_patience = 10
early_stopping_counter = 0
best_valid_loss = float('inf')

# Training loop for 100 epochs (with early stopping)
num_epochs = 100


In [ ]:
for epoch in range(num_epochs):
    vit_model.train()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        #print("this loop is running")

        optimizer.zero_grad()
        outputs = vit_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Evaluate the model on the training set
    vit_model.eval()
    train_correct = 0
    train_total = 0

    with torch.no_grad():
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = vit_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            #print("training loop running")
            train_correct += (predicted == labels).sum().item()

    train_accuracy = train_correct / train_total
    print(f'Train Accuracy: {train_accuracy:.4f}')

    # Evaluate the model on the validation set
    vit_model.eval()
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = vit_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            valid_total += labels.size(0)
            #print("valid loop running")
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = valid_correct / valid_total
    print(f'Valid Accuracy: {valid_accuracy:.4f}')

    # Save the model with the best accuracy on the validation set
    if valid_accuracy > best_accuracy_vit:
        best_accuracy_vit = valid_accuracy
        best_model_vit = vit_model.state_dict()
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    # Store accuracies for plotting
    train_accuracies.append(train_accuracy)
    valid_accuracies.append(valid_accuracy)

    # Print and save model checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Accuracy: {train_accuracy:.4f}, Valid Accuracy: {valid_accuracy:.4f}')

    # Early stopping
    if early_stopping_counter >= early_stopping_patience:
        print("Early stopping! No improvement for {} epochs.".format(early_stopping_patience))
        break



In [ ]:
# Plot the accuracy change with epochs
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, len(valid_accuracies) + 1), valid_accuracies, label='Valid Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.legend()
plt.show()

Comparing to the other curves, we can see the validation accuracy doing significantly well comparing to the training accuracy

In [ ]:
# Load the best model and evaluate it on the test set
vit_model.load_state_dict(best_model_vit)
vit_model.eval()
test_correct = 0
test_total = 0
all_labels = []
all_predictions = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = vit_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(torch.nn.functional.softmax(outputs, dim=1)[:, 1].cpu().numpy())

test_accuracy_vit = test_correct / test_total
print(f'Best Accuracy on Test Set (ResNet): {100 * test_accuracy_vit:.2f}%')

# Convert the lists to numpy arrays
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# Plot ROC Curve for ResNet
fpr_vit, tpr_vit, thresholds_vit = roc_curve(all_labels, all_predictions)
roc_auc_vit = auc(fpr_vit, tpr_vit)

plt.figure()
plt.plot(fpr_vit, tpr_vit, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_vit:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve (ViT)')
plt.legend(loc="lower right")
plt.show()

# Plot Confusion Matrix for ResNet
cm_vit = confusion_matrix(all_labels, (all_predictions > 0.5).astype(int))
plt.figure(figsize=(5, 4))
sns.heatmap(cm_vit, annot=True, fmt="d", cmap="Blues", xticklabels=["No DR", "DR"], yticklabels=["No DR", "DR"])
plt.title("Confusion Matrix (ViT)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


In [ ]:
vit_sensitivity = 110/112
vit_specificity = 116/119

In [ ]:
print(test_accuracy_resnet)
print(test_accuracy_effnet)
print(test_accuracy_vit)

In [ ]:
# Variables and their corresponding test accuracies
models = ['ResNet50', 'EfficientNet-b2', 'ViT']
test_accuracies = [test_accuracy_resnet, test_accuracy_effnet, test_accuracy_mbnet, test_accuracy_vit]

# Plotting the bar chart
plt.bar(models, test_accuracies, color=['blue', 'green', 'red'])
plt.xlabel('Models')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy of Different Models')
plt.ylim(0, 1)  # Set the y-axis range from 0 to 1 (assuming accuracy values are in the range 0 to 1)

# Display the test accuracy values on top of the bars
for i, accuracy in enumerate(test_accuracies):
    plt.text(i, accuracy + 0.01, f'{accuracy:.2f}', ha='center', va='bottom')

# Show the plot
plt.show()


based on the test accuracy results for different pre-trained models:

Comparing Performance of Various Pre-trained Models for DR Diagnosis:

The test accuracies for the different models are as follows:

ResNet50: 95.67%
EfficientNet-b2: 97.83%
MobileNet V3: 97.83%
ViT (Vision Transformer): 97.83%
These accuracy values suggest that all models perform very well in diagnosing diabetic retinopathy (DR). The test accuracies are quite high, with EfficientNet-b2, MobileNet V3, and ViT achieving the highest accuracy of 97.83%.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sensitivity and Specificity values
models = ['ResNet50', 'EfficientNet-b2', 'ViT']
sensitivity = [106/109, 110/112, 110/112]
specificity = [115/122, 116/119, 116/119]

# Plotting the bar charts
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

# Bar chart for Sensitivity
axes[0].bar(models, sensitivity, color=['blue', 'green', 'red'])
axes[0].set_title('Sensitivity of Different Models')
axes[0].set_ylabel('Sensitivity')

# Bar chart for Specificity
axes[1].bar(models, specificity, color=['blue', 'green', 'red'])
axes[1].set_title('Specificity of Different Models')
axes[1].set_ylabel('Specificity')

# Display the values on top of the bars
for ax in axes:
    for i, val in enumerate(ax.patches):
        ax.text(val.get_x() + val.get_width() / 2, val.get_height() + 0.01,
                f'{val.get_height():.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import timm  # For EfficientNet and ViT

class HybridDRClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridDRClassifier, self).__init__()

        # Stage 1: Load and trim pretrained models
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        self.effnet = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)

        # Stage 2: Combine feature dimensions
        self.feature_dim = 2048 + 1408 + 768

        # Stage 3: Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Stage 4–6
        resnet_feat = self.resnet(x)
        effnet_feat = self.effnet(x)
        vit_feat = self.vit(x)
        combined = torch.cat((resnet_feat, effnet_feat, vit_feat), dim=1)
        out = self.classifier(combined)
        return out

# Inference helper
def diagnose_dr(model, image_tensor):
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
    diagnosis_labels = [
        "No Diabetic Retinopathy",
        "Mild (Stage 1)",
        "Moderate (Stage 2)",
        "Severe (Stage 3)",
        "Proliferative DR (Stage 4)"
    ]
    return diagnosis_labels[predicted.item()]

# Save model after training
if __name__ == "__main__":
    model = HybridDRClassifier(num_classes=5)
    torch.save(model.state_dict(), 'hybrid_dr_model.pth')
    print("✅ Model saved as 'hybrid_dr_model.pth'")

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import torch
from torchvision import transforms
from PIL import Image

# Load the trained model
model = HybridDRClassifier(num_classes=5)
model.load_state_dict(torch.load('hybrid_dr_model.pth', map_location=torch.device('cpu')))
model.eval()

# Title
st.title("PolyVision: DR Stage Prediction")
st.write("Upload an ultra-widefield retinal image to check for Diabetic Retinopathy stage.")

# Upload image
uploaded_file = st.file_uploader("Choose a fundus image", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded Image', use_column_width=True)

    # Preprocessing
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    image_tensor = preprocess(image).unsqueeze(0)  # Add batch dimension

    # Prediction
    with st.spinner('Diagnosing...'):
        prediction = diagnose_dr(model, image_tensor)
        st.success(f"Diagnosis: {prediction}")


In [ ]:
!streamlit run /usr/local/lib/python3.10/dist-packages/colab_kernel_launcher.py